# Case 3 — Bridge Structural Monitoring: Faulty Sensor Spikes

## Assignment context

This notebook is part of the Smart Monitoring System assignment for Master students in Civil Engineering and Territorial Protection.

**Monitoring objective:** Bridge structural health monitoring  
**Main issue:** Faulty sensor spikes and outliers

Tasks:

1. inspect the raw sensor data;
2. detect the issue visually;
3. detect the issue statistically or with ML;
4. decide whether to correct, flag or preserve the observations;
5. prepare the dataset for ingestion, analysis, dashboarding and alerting in istSOS4Things.


## Case description

A vibration sensor is installed on a bridge to monitor structural behaviour. The system should detect abnormal vibration levels, but the sensor sometimes produces unrealistic spikes due to malfunction.

Students should distinguish isolated faulty spikes from potentially meaningful events.


## 1. Import libraries and load the dataset


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = "dataset_3_bridge_faulty_sensor.csv"
VALUE_COL = "vibration_g"

df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)
df.head()


## 2. First inspection


In [ ]:
print(df.info())
print("\nMissing values:")
print(df.isna().sum())
print("\nSummary statistics:")
display(df.describe())


## 3. Visual inspection of raw data


In [ ]:
plt.figure(figsize=(12,4))
plt.plot(df["timestamp"], df[VALUE_COL], marker=".", linewidth=1)
plt.title(f"Raw time series: {VALUE_COL}")
plt.xlabel("Time")
plt.ylabel(VALUE_COL)
plt.grid(True)
plt.show()


## Detection and correction strategy

Suggested checks:

- time-series plot;
- histogram and boxplot;
- z-score;
- IQR method;
- optional Isolation Forest.

Possible treatment:

- replace faulty spikes with NaN;
- interpolate short isolated gaps only if justified;
- preserve a quality flag.


## 4. Histogram and boxplot


In [ ]:
plt.figure(figsize=(8,4))
plt.hist(df[VALUE_COL].dropna(), bins=40)
plt.title("Distribution of vibration values")
plt.xlabel("Vibration [g]")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()

plt.figure(figsize=(8,3))
plt.boxplot(df[VALUE_COL].dropna(), vert=False)
plt.title("Boxplot of vibration values")
plt.xlabel("Vibration [g]")
plt.grid(True)
plt.show()


## 5. Detect outliers with z-score and IQR


In [ ]:
mean = df[VALUE_COL].mean()
std = df[VALUE_COL].std()
df["z_score"] = (df[VALUE_COL] - mean) / std
df["z_outlier"] = df["z_score"].abs() > 3

q1 = df[VALUE_COL].quantile(0.25)
q3 = df[VALUE_COL].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
df["iqr_outlier"] = (df[VALUE_COL] < lower) | (df[VALUE_COL] > upper)

print("Z-score outliers:", df["z_outlier"].sum())
print("IQR outliers:", df["iqr_outlier"].sum())

plt.figure(figsize=(12,4))
plt.plot(df["timestamp"], df[VALUE_COL], label="raw", linewidth=1)
plt.scatter(df.loc[df["iqr_outlier"], "timestamp"], df.loc[df["iqr_outlier"], VALUE_COL], label="IQR outliers")
plt.title("Detected vibration outliers")
plt.xlabel("Time")
plt.ylabel("Vibration [g]")
plt.legend()
plt.grid(True)
plt.show()


## 6. Optional ML detection with Isolation Forest


In [ ]:
from sklearn.ensemble import IsolationForest

X = df[[VALUE_COL]].dropna()
model = IsolationForest(contamination=0.05, random_state=42)
labels = model.fit_predict(X)

df["iforest_outlier"] = False
df.loc[X.index, "iforest_outlier"] = labels == -1

plt.figure(figsize=(12,4))
plt.plot(df["timestamp"], df[VALUE_COL], label="raw", linewidth=1)
plt.scatter(df.loc[df["iforest_outlier"], "timestamp"], df.loc[df["iforest_outlier"], VALUE_COL], label="Isolation Forest outliers")
plt.title("Isolation Forest outlier detection")
plt.xlabel("Time")
plt.ylabel("Vibration [g]")
plt.legend()
plt.grid(True)
plt.show()


## 7. Clean faulty spikes and export


In [ ]:
df["vibration_cleaned_g"] = df[VALUE_COL]
df.loc[df["iqr_outlier"], "vibration_cleaned_g"] = np.nan
df["vibration_cleaned_g"] = df["vibration_cleaned_g"].interpolate(method="linear")
df["quality_flag"] = np.where(df["iqr_outlier"], "faulty_spike_corrected", "raw")

plt.figure(figsize=(12,4))
plt.plot(df["timestamp"], df[VALUE_COL], label="raw", alpha=0.5)
plt.plot(df["timestamp"], df["vibration_cleaned_g"], label="cleaned", linewidth=2)
plt.title("Raw vs cleaned vibration data")
plt.xlabel("Time")
plt.ylabel("Vibration [g]")
plt.legend()
plt.grid(True)
plt.show()

cleaned = df[["timestamp", "vibration_g", "vibration_cleaned_g", "quality_flag"]]
cleaned.to_csv("cleaned_dataset_3_bridge_faulty_sensor.csv", index=False)
cleaned.head()


## Final questions

1. What is the main data quality issue or hazardous event?
2. Which visual method was most useful?
3. Which statistical or ML method was most useful?
4. Which observations should be corrected, removed, flagged or preserved?
5. What would be a suitable alerting rule for an operational dashboard?
6. How would you model this dataset in SensorThings API?
   - Thing
   - Location
   - Sensor
   - ObservedProperty
   - Datastream
   - Observation
